# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/latent-play/blob/main/notebooks/colab_training_and_demo.ipynb)

Este notebook ejecuta el flujo completo de **`latent-play`** en Google Colab Pro (**A100 GPU + High-RAM**) con **persistencia automática en Google Drive cada 5 minutos**:
- Si Colab se desconecta en cualquier momento, al volver a darle a **Run all** retomará automáticamente el entrenamiento desde el último checkpoint guardado en tu Google Drive (`MyDrive/latent_play_runs/`).
- Los datasets (`data/`) se guardan en el disco NVMe local de Colab para máxima velocidad de lectura (`np.memmap`), mientras que `checkpoints/` y `outputs/` están enlazados directamente a tu Google Drive.

## 1. Setup, Montaje de Google Drive y Clonado del Repo

In [ ]:
import os, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# 1. Clonar o actualizar el repositorio en /content/latent-play
import os
if not os.path.exists("/content/latent-play"):
    !git clone https://github.com/rivas-vega/latent-play.git /content/latent-play
%cd /content/latent-play
!git pull origin main

# 2. Montar Google Drive y enlazar checkpoints/ y outputs/ para no perder nada si se desconecta
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/latent_play_runs"
    os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
    for folder in ("checkpoints", "outputs"):
        if os.path.islink(folder):
            os.unlink(folder)
        elif os.path.isdir(folder):
            !cp -rn {folder}/* {DRIVE_ROOT}/{folder}/ 2>/dev/null || true
            !rm -rf {folder}
        os.symlink(f"{DRIVE_ROOT}/{folder}", folder)
    print(f"Checkpoints y outputs enlazados a Google Drive: {DRIVE_ROOT}")
except Exception as e:
    print(f"Ejecutando sin Google Drive ({e}): guardando en disco local.")

# 3. Instalar dependencias y pasar los 41 tests unitarios
!pip install -q -e ".[crafter,dev]" matplotlib
!pytest -q

## 2. Fase A — Validación Rápida en `ToyGridEnv` (~3 min en A100)
Verifica en ~3 minutos el pipeline completo sobre `ToyGridEnv` antes de pasar a Crafter.

In [ ]:
!if [ ! -f data/toy/meta.json ]; then python scripts/collect_data.py --config configs/collect_toy.yaml --total-steps 100000 --workers 4; fi
!python scripts/train_lam.py --config configs/lam_toy.yaml train.amp=auto
!python scripts/train_dyn.py --config configs/dyn_toy.yaml lam_checkpoint=checkpoints/lam_toy/lam_final.pt train.out_dir=checkpoints/dyn_toy_lam train.amp=auto
!python scripts/train_dyn.py --config configs/dyn_toy.yaml train.out_dir=checkpoints/dyn_toy_oracle train.amp=auto
!python scripts/sweep_labels.py --data-root data/toy --lam-ckpt checkpoints/lam_toy/lam_final.pt --dyn-ckpt checkpoints/dyn_toy_lam/dyn_final.pt --oracle-ckpt checkpoints/dyn_toy_oracle/dyn_final.pt --fractions 0.001 0.01 0.05 0.1 1.0 --num-episodes 25 --out-dir outputs/sweep_toy

## 3. Fase B — Recolección de Datos en `Crafter` (300k pasos)

In [ ]:
!if [ ! -f data/crafter/meta.json ]; then python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps 300000 --workers 4; fi
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif

## 4. Entrenar Latent Action Model (LAM) en `Crafter` (`label_fraction=0.0`)
Guarda checkpoints automáticamente cada 5 minutos en tu Google Drive (`MyDrive/latent_play_runs/checkpoints/lam_crafter`).

In [ ]:
!python scripts/train_lam.py --config configs/lam_crafter.yaml train.compile=true train.amp=auto

## 5. Entrenar JEPA Dynamics + SIGReg en `Crafter` (Latent Actions vs. Oracle)
Guarda checkpoints automáticamente cada 5 minutos en tu Google Drive (`dyn_crafter_lam` y `dyn_crafter_oracle`).

In [ ]:
# 5a. Nuestro modelo (latent-play): condicionado únicamente en códigos latentes del LAM
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.out_dir=checkpoints/dyn_crafter_lam train.amp=auto

# 5b. Oráculo (Upper Bound): condicionado en el 100% de las acciones verdaderas de Crafter
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml train.out_dir=checkpoints/dyn_crafter_oracle train.amp=auto

## 6. Barrido de Fracción de Etiquetas (0.1% a 100%), Planificación CEM y VoE Scorer

In [ ]:
!python scripts/sweep_labels.py \
    --data-root data/crafter \
    --lam-ckpt checkpoints/lam_crafter/lam_final.pt \
    --dyn-ckpt checkpoints/dyn_crafter_lam/dyn_final.pt \
    --oracle-ckpt checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --fractions 0.001 0.01 0.05 0.1 1.0 \
    --num-episodes 30 \
    --out-dir outputs/sweep_results

!python scripts/eval_voe.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --lam-checkpoint checkpoints/lam_crafter/lam_final.pt \
    --out outputs/voe_metrics.json

!python scripts/plot_paper_figures.py --metrics-json outputs/sweep_results/sweep_metrics.json --out-dir outputs/figures

In [ ]:
!python scripts/eval_planning.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --alignment-checkpoint outputs/sweep_results/alignment_f0.01_s0.pt \
    --num-episodes 20

from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")

## 7. Empaquetar Resultados y Logs para Descargar

In [ ]:
!zip -r latent_play_results.zip outputs/ checkpoints/*/log.jsonl checkpoints/*/confusion.json checkpoints/*/config.json
from google.colab import files
files.download("latent_play_results.zip")